# BirdCLEF+ 2026 — Perch v2 Logit Direct Submission

Perch v2 TFLite の **logit 出力を直接 sigmoid** して予測する提出用ノートブック。

## Input
- `birdclef-2026` (Competition)
- Training notebook output (`perch_v2.tflite` + `species_to_perch_idx.pkl`)

## Settings
| Setting | Value |
|---------|-------|
| Accelerator | None (CPU) |
| Internet | **OFF** |

In [ ]:
import os, glob, pickle, collections, time, warnings
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf

warnings.filterwarnings('ignore')
print(f'TensorFlow: {tf.__version__}')

In [ ]:
# ============================================================
# Path + Constants
# ============================================================
COMP_DATA = None
for p in ['/kaggle/input/birdclef-2026', '/kaggle/input/competitions/birdclef-2026']:
    if os.path.exists(p): COMP_DATA = p; break
assert COMP_DATA

# TFLite model (from training notebook output)
TFLITE_PATH = None
for f in glob.glob('/kaggle/input/**/*.tflite', recursive=True): TFLITE_PATH = f; break
assert TFLITE_PATH, 'perch_v2.tflite not found'

# Species→Perch logit index mapping (from training notebook output)
MAPPING_PATH = None
for f in glob.glob('/kaggle/input/**/*perch_idx*.pkl', recursive=True): MAPPING_PATH = f; break
assert MAPPING_PATH, 'species_to_perch_idx.pkl not found'

SAMPLE_RATE = 32000      # Hz: Perch v2 のサンプリング周波数
DURATION = 5             # 秒: Perch v2 の入力窓長（5秒固定）
WINDOW_SAMPLES = SAMPLE_RATE * DURATION  # = 160,000 samples/window
OUTPUT_DIR = '/kaggle/working'

print(f'Data:    {COMP_DATA}')
print(f'TFLite:  {TFLITE_PATH}')
print(f'Mapping: {MAPPING_PATH}')

In [ ]:
# ============================================================
# Load TFLite Perch — logit output を使う
# ============================================================
interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH, num_threads=4)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# テスト実行して各出力の shape を確認
interpreter.set_tensor(input_details[0]['index'], np.zeros((1, WINDOW_SAMPLES), dtype=np.float32))
interpreter.invoke()

# logit 出力 (~14,795次元) のインデックスを特定
LOGIT_IDX = None
for od in output_details:
    r = interpreter.get_tensor(od['index'])
    print(f"  {od['name']}: shape={r.shape}")
    if r.shape[-1] > 5000:  # logits は ~14,795
        LOGIT_IDX = od['index']

assert LOGIT_IDX is not None, 'Logit output not found in TFLite'

def get_logits(waveform):
    """5秒波形 → Perch logit (14795,) を返す。"""
    interpreter.set_tensor(input_details[0]['index'], waveform.reshape(1, -1).astype(np.float32))
    interpreter.invoke()
    return interpreter.get_tensor(LOGIT_IDX).flatten().astype(np.float32)

# 動作確認
test_logits = get_logits(np.zeros(WINDOW_SAMPLES, dtype=np.float32))
print(f'\nLogit shape: {test_logits.shape}')
print(f'Logit range: [{test_logits.min():.2f}, {test_logits.max():.2f}]')
print('TFLite Perch logit OK')

In [ ]:
# ============================================================
# Load mapping + submission info
# ============================================================
with open(MAPPING_PATH, 'rb') as f:
    species_to_perch_idx = pickle.load(f)

sample_sub = pd.read_csv(f'{COMP_DATA}/sample_submission.csv')
species_cols = [c for c in sample_sub.columns if c != 'row_id']

print(f'Mapped species:   {len(species_to_perch_idx)} / {len(species_cols)}')
print(f'Unmapped species: {len(species_cols) - len(species_to_perch_idx)}')

In [ ]:
# ============================================================
# Inference — logit を sigmoid して直接予測
# ============================================================
def sigmoid(x):
    """数値安定な sigmoid。logit → [0, 1] の確率に変換。"""
    return 1 / (1 + np.exp(-np.clip(x, -50, 50)))

test_dir = f'{COMP_DATA}/test_soundscapes'
test_files = sorted(f for f in os.listdir(test_dir) if f.endswith('.ogg'))
print(f'Test files: {len(test_files)}')

rows = []
t0 = time.time()
for fi, fname in enumerate(test_files):
    stem = os.path.splitext(fname)[0]
    full_wf, _ = librosa.load(os.path.join(test_dir, fname), sr=SAMPLE_RATE, mono=True)
    if len(full_wf) == 0: continue
    
    # 5秒窓で非重複スライド
    for start in range(0, len(full_wf), WINDOW_SAMPLES):
        window = full_wf[start:start + WINDOW_SAMPLES]
        if len(window) < WINDOW_SAMPLES:
            window = np.pad(window, (0, WINDOW_SAMPLES - len(window)))
        
        logits = get_logits(window.astype(np.float32))
        probs = sigmoid(logits)
        
        bucket_sec = (start // WINDOW_SAMPLES) * DURATION
        row = {'row_id': f'{stem}_{bucket_sec}'}
        for sp in species_cols:
            if sp in species_to_perch_idx:
                row[sp] = float(probs[species_to_perch_idx[sp]])
            else:
                row[sp] = 0.0
        rows.append(row)
    
    if (fi + 1) % 50 == 0:
        print(f'  [{fi+1}/{len(test_files)}] {int(time.time()-t0)}s', flush=True)

print(f'Done: {len(rows)} predictions, {int(time.time()-t0)}s')

In [ ]:
# ============================================================
# Build + save submission
# ============================================================
submission = pd.DataFrame(rows, columns=['row_id'] + species_cols)
submission = submission.set_index('row_id').reindex(sample_sub['row_id'], fill_value=0.0).reset_index()
submission.to_csv(f'{OUTPUT_DIR}/submission.csv', index=False)

assert len(submission) == len(sample_sub)
assert list(submission.columns) == list(sample_sub.columns)
assert submission.isnull().sum().sum() == 0

vals = submission.drop('row_id', axis=1).values
print(f'Submission saved')
print(f'  Rows:             {len(submission)}')
print(f'  Mean prediction:  {vals.mean():.6f}')
print(f'  Max prediction:   {vals.max():.4f}')
print(f'  Zero rate:        {(vals == 0).mean():.2%}')
print(f'  Non-zero species: {(vals.max(axis=0) > 0).sum()} / {len(species_cols)}')